<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_04_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 04 — The Report

**Paired with L7.1 · Fundamentals of PINNs, and L7.2 · Fundamental PDEs**

Everything the set measured, in one table and one figure; a current of your
own; five questions in your own words; your answers from notebooks 01 to 03;
and one question across all of them to conclude. The cells at the end turn it
into a PDF for Moodle.

This notebook needs the files notebooks 01, 02 and 03 saved. Run those first.

---

## 0 · Setup

**What these cells do.** Fetch the library files, find your saved results, import the modules and load the three results files.

**How.** `cc.needed` checks that `nb01_truth.npz`, `nb02_pinn.npz` and `nb03_helmholtz.npz` are there — on Colab without Drive it asks you to upload them.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex07.1_outputs")

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

cc.needed("nb01_truth.npz", "nb02_pinn.npz", "nb03_helmholtz.npz")   # asks for any missing
r1 = np.load(os.path.join(cc.OUTPUT_DIR, "nb01_truth.npz"))
r2 = np.load(os.path.join(cc.OUTPUT_DIR, "nb02_pinn.npz"))
r3 = np.load(os.path.join(cc.OUTPUT_DIR, "nb03_helmholtz.npz"))
print("loaded the results of notebooks 01, 02 and 03")

---

## 1 · The results

**What this cell does.** Gathers the set's numbers into one table, which goes into your report as it stands, and draws the figure the report carries: accuracy against time for every method.

**How.** It reads the three results files. The figure puts each finite-difference grid and the hard network at their worst error and their time per current; the network's training time is printed beside it, because the figure cannot show it.

In [ ]:
peaks = [f.max() for f in r1["fields"]]
rated, top = peaks[3], peaks[-1]
i2 = rated * (r1["currents"][-1] / r1["currents"][3]) ** 2          # what fixed heat would give
sweep = dict(zip(r2["sweep_names"], r2["sweep"]))
RESULTS_MD = "\n".join([
    "| | |", "|---|---|",
    f"| hot spot at rated current, ground truth | {rated:.2f} K above the wall |",
    f"| hot spot at 45 A, against $I^2$ scaling | {top:.2f} K, {100 * (top / i2 - 1):+.1f} % |",
    f"| PINN, soft walls: worst error / on the walls | {float(r2['worst_soft']):.3f} K / {float(r2['wall_soft']):.3f} K |",
    f"| PINN, hard walls: worst error | {float(r2['worst_hard']):.3f} K |",
    *[f"| hard network {k}: parameters, worst error | {int(v[0])}, {v[1]:.3f} K |" for k, v in sweep.items()],
    f"| finite differences as accurate: grid, time per current | {int(r2['fdm_grid'])} x {2 * int(r2['fdm_grid']) - 1}, {1e3 * float(r2['t_fdm']):.2f} ms |",
    f"| trained network: time per current, training | {1e3 * float(r2['t_net']):.2f} ms, {float(r2['train_hard']):.0f} s |",
    f"| currents before the training pays | {float(r2['break_even']):,.0f} |",
    f"| slot's first eigenvalue: formula / network | {float(r3['lam1']) / 1e6:.5f} / {float(r3['lam1_pinn']) / 1e6:.5f} per mm$^2$ |",
    f"| thermal runaway / class F reached | {float(r3['I_star']):.0f} A / {float(r3['class_limits'][1]):.0f} A |",
])
print(RESULTS_MD)

fig, ax = plt.subplots(figsize=(6.6, 4.3))
ax.loglog(1e3 * r1["times"], r1["errors"], "o-", color=CYCLE[1], lw=2, label="finite differences")
for g, t, e in zip(r1["grids"], r1["times"], r1["errors"]):
    ax.annotate(f" {g}", (1e3 * t, e), fontsize=8, color=CYCLE[1])
ax.loglog(1e3 * float(r2["t_net"]), float(r2["worst_hard"]), "s", ms=9, color=CYCLE[0],
          label=f"PINN, hard walls (+{float(r2['train_hard']):.0f} s training)")
ax.set_xlabel("time per current  [ms]"); ax.set_ylabel("worst error  [K]")
ax.set_title("accuracy against time"); ax.legend(frameon=False)
plt.savefig(cc.output_path("Ex07.1_report.png"), dpi=150, bbox_inches="tight")
plt.show()

---

## 2 · Your own current

**What this cell does.** Draws a current of your own from your study number, and works out the slot at it: the hot spot, the feedback ratio and how much hot copper pushes the temperature up.

**How.** `personal_seed` turns your study number into a seed, the seed into a current between 10 and 45 A. Your supervisor can regenerate the same numbers from your study number alone. Question 3 below asks you to quote them.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
I_you = float(np.random.default_rng(SEED).uniform(*pb.I_RANGE))
ratio = (I_you / float(r3["I_star"])) ** 2                  # feedback, against the eigenvalue
hot = pb.fdm_solve(I_you, 41)[2].max()
print(f"study number : {STUDENT_NUMBER}   seed : {SEED}")
print(f"your current : {I_you:.2f} A per wire, {I_you / pb.A_WIRE / 1e6:.2f} A/mm^2")
print(f"hot spot     : {hot:.3f} K above the wall, {pb.T_WALL + hot:.1f} C")
print(f"feedback     : {ratio:.2%} of the eigenvalue; the rise is pushed up about {1 / (1 - ratio) - 1:.1%}")

---

## 3 · Your answers

Replace every string. Keep to the word limits; they are tight on purpose.

In [ ]:
# TODO: write your report. Every string below must be replaced.

NAME = "Your Name"
GROUP = "your group"

Q1_WINNER = """
(150 words) Report who won, on accuracy and on time per current, with your
numbers: the worst errors of the soft and hard networks, the finite-difference
grid that matched the hard one and its time, and the number of currents before
the training paid. Say for which job you would choose each method.
"""

Q2_WALLS = """
(120 words) The soft network missed the walls by a fraction of a kelvin and the
inside of the slot by more than a kelvin. Explain how an error on the wall
reaches the middle of the slot, and why the mask removes it.
"""

Q3_FEEDBACK = """
(120 words) Quote your current, your hot spot and your feedback ratio. Explain
in plain words why hot copper turns the slot's Poisson equation into a
Helmholtz equation, and what the first eigenvalue tells a designer.
"""

Q4_LEVERS = """
(100 words) Choose one lever to run this winding at 45 A below class F, and
say what it costs the machine.
"""

Q5_DISTRUST = """
(100 words) Which number in your results do you trust least, and what would
you check to trust it more?
"""

---

## 4 · Check, assemble, save

**What this cell does.** Checks every answer against its word limit, and writes `Ex07.1_report.md` with your name, the results table and your answers.

**How.** An answer still holding its prompt, too long or far too short is listed instead of written.

In [ ]:
answers = {
    "1 · Who won": (Q1_WINNER, 150),
    "2 · Why a soft wall leaks": (Q2_WALLS, 120),
    "3 · Hot copper and Helmholtz": (Q3_FEEDBACK, 120),
    "4 · Keeping it cool": (Q4_LEVERS, 100),
    "5 · What you distrust": (Q5_DISTRUST, 100),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    out = ["# Ex_07.1 — Fundamentals of PINNs: a stator slot", "",
           f"**{NAME}** · {GROUP}", "",
           f"study number {STUDENT_NUMBER} · seed {SEED} · your current {I_you:.2f} A", "",
           "Deep Learning for Engineering · Aalborg University · 2026", "",
           "---", "", "## Results", "", RESULTS_MD, ""]
    for title, (text, _) in answers.items():
        out += [f"## {title}", "", text.strip(), ""]
    path = cc.output_path("Ex07.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready:` and a list, until every answer is yours;
then the path of the report.

---

## The questions from notebooks 01 to 03

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all of them, each under its question, to the end of
the report. The last one is not from a notebook: it is the question across all
of them, and it concludes the report. On Colab every notebook runs on its own
machine, so this notebook cannot read what you wrote in the others: copying is
the only way across.

The arrow after each question names the lecture question it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 03 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · The Ground Truth, by Finite Differences -----------------
    # 01.1 Finite differences turned the slot into one linear system. What are
    # its unknowns, what did the grid do that a physics-informed network does
    # without, and how many unknowns did your finest grid have? (-> L7.1 Q4)
    "01.1": """
""",
    # 01.2 The error fell four times with every halving of $h$. Say why, and
    # how Richardson extrapolation uses that to build a reference better than
    # either grid. Why must a reference be far more accurate than what it
    # judges? (-> L7.1 Q4)
    "01.2": """
""",
    # 01.3 The hot spot grew faster than $I^2$. Explain why from the heat
    # source, and why each current is nevertheless a single linear solve. What
    # makes the problem well posed at 45 A, and what would stop it being so?
    # (-> L7.1 Q5)
    "01.3": """
""",
    # 01.4 Which grid would you choose to know the hot spot to 0.01 K, and how
    # long did it take? Keep that number: it is what notebook 02 has to beat.
    # (-> L7.1 Q4)
    "01.4": """
""",

    # ---- notebook 02 · The Walls, Soft and Hard --------------------------------
    # 02.1 The soft network was about 0.2 K off on the walls and more than a
    # kelvin off inside. Explain from the loss why a soft wall is met only
    # nearly, and why its error does not stay on the wall. (-> L7.1 Q7)
    "02.1": """
""",
    # 02.2 What does the mask guarantee, from the first step of training? What
    # would it cost for a slot with a rounded bottom, or a wall held at 80 °C
    # instead of the iron's temperature? (-> L7.1 Q9)
    "02.2": """
""",
    # 02.3 The network trained on 2000 points inside the slot. How does that
    # compare with $N^*$ for four layers of 32 with three inputs, and what
    # happened to the accuracy as the network shrank? (-> L7.1 Q2)
    "02.3": """
""",
    # 02.4 Who won, and by how much? For which job would you train a network
    # instead of solving with finite differences, and how many currents would
    # it take before the training paid? (-> L7.1 Q4)
    "02.4": """
""",

    # ---- notebook 03 · Hot Copper, and the Helmholtz Equation ------------------
    # 03.1 Explain in your own words why hot copper turns the slot's Poisson
    # equation into a Helmholtz equation. What does $\lambda = c_1/k$ measure,
    # and what does $\lambda_1$ measure? (-> L7.2 Q5)
    "03.1": """
""",
    # 03.2 What happens to the temperature as the feedback ratio approaches 1,
    # and why does a real machine never get there? What does that say about
    # when the problem is well posed? (-> L7.1 Q5)
    "03.2": """
""",
    # 03.3 Without the norm term the network returned $u \equiv 0$. Why is
    # that a perfect solution of the residual, and what does each of the two
    # safeguards exclude? (-> L7.2 Q3)
    "03.3": """
""",
    # 03.4 Which lever would you pull to run this winding at 45 A below class
    # F, and what does it cost the machine? (-> L7.2 Q5)
    "03.4": """
""",

    # ---- to conclude, across all the notebooks ---------------------------------
    # C Finite differences won this slot. For which problem would you train a
    # physics-informed network instead, and what would you check it against?
    # (-> L7.1, Exercise slide)
    "C": """
""",
}

In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
NOTEBOOK_QUESTIONS = {
    '01.1': ('01', 'The Ground Truth, by Finite Differences', 'Finite differences turned the slot into one linear system. What are its unknowns, what did the grid do that a physics-informed network does without, and how many unknowns did your finest grid have?', 'L7.1 Q4'),
    '01.2': ('01', 'The Ground Truth, by Finite Differences', 'The error fell four times with every halving of $h$. Say why, and how Richardson extrapolation uses that to build a reference better than either grid. Why must a reference be far more accurate than what it judges?', 'L7.1 Q4'),
    '01.3': ('01', 'The Ground Truth, by Finite Differences', 'The hot spot grew faster than $I^2$. Explain why from the heat source, and why each current is nevertheless a single linear solve. What makes the problem well posed at 45 A, and what would stop it being so?', 'L7.1 Q5'),
    '01.4': ('01', 'The Ground Truth, by Finite Differences', 'Which grid would you choose to know the hot spot to 0.01 K, and how long did it take? Keep that number: it is what notebook 02 has to beat.', 'L7.1 Q4'),
    '02.1': ('02', 'The Walls, Soft and Hard', 'The soft network was about 0.2 K off on the walls and more than a kelvin off inside. Explain from the loss why a soft wall is met only nearly, and why its error does not stay on the wall.', 'L7.1 Q7'),
    '02.2': ('02', 'The Walls, Soft and Hard', "What does the mask guarantee, from the first step of training? What would it cost for a slot with a rounded bottom, or a wall held at 80 °C instead of the iron's temperature?", 'L7.1 Q9'),
    '02.3': ('02', 'The Walls, Soft and Hard', 'The network trained on 2000 points inside the slot. How does that compare with $N^*$ for four layers of 32 with three inputs, and what happened to the accuracy as the network shrank?', 'L7.1 Q2'),
    '02.4': ('02', 'The Walls, Soft and Hard', 'Who won, and by how much? For which job would you train a network instead of solving with finite differences, and how many currents would it take before the training paid?', 'L7.1 Q4'),
    '03.1': ('03', 'Hot Copper, and the Helmholtz Equation', "Explain in your own words why hot copper turns the slot's Poisson equation into a Helmholtz equation. What does $\\lambda = c_1/k$ measure, and what does $\\lambda_1$ measure?", 'L7.2 Q5'),
    '03.2': ('03', 'Hot Copper, and the Helmholtz Equation', 'What happens to the temperature as the feedback ratio approaches 1, and why does a real machine never get there? What does that say about when the problem is well posed?', 'L7.1 Q5'),
    '03.3': ('03', 'Hot Copper, and the Helmholtz Equation', 'Without the norm term the network returned $u \\equiv 0$. Why is that a perfect solution of the residual, and what does each of the two safeguards exclude?', 'L7.2 Q3'),
    '03.4': ('03', 'Hot Copper, and the Helmholtz Equation', 'Which lever would you pull to run this winding at 45 A below class F, and what does it cost the machine?', 'L7.2 Q5'),
    'C': ('C', 'to conclude', 'Finite differences won this slot. For which problem would you train a physics-informed network instead, and what would you check it against?', 'L7.1, Exercise slide'),
}

report_md = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex07.1_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "", "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "", answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))

**What you should see.** `added to .../Ex07.1_report.md: 13 of 13 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run it again after any change
to the report above it, because rewriting the report removes the section; then
make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
turns the report into a PDF, with the figure from section 1, and downloads it.
**Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.1_report.md into Ex07.1_report.pdf, with any figure
# saved as Ex07.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 5 · What Ex_07.1 was for

One slot, solved three ways, and one equation that turned out to be two.

* **A reference must be far better than what it judges.** Notebook 01 built one
  good to a hundred-thousandth of a kelvin, and showed it had converged.
* **A wall in the loss is a request; a wall in the function is a fact.** The
  soft network traded its wall against the equation and paid for it
  everywhere inside. Build the condition in when you can write the mask.
* **A network pays for its training only when it is asked many questions.**
  For one slot at one current, finite differences are cheaper and more
  accurate. Know the break-even before you choose.
* **Physics can change the equation.** Hot copper made the slot's Poisson
  equation a Helmholtz equation, and its first eigenvalue said how much the
  temperature is pushed up, and where the model stops having an answer.

---

## 6 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**: the course builds it with a classical
solver once you have chosen the project, and shows that it has converged. You
do not solve the physics a second way; the month goes into the deep learning -
making a network predict what the ground truth says, and showing how well it
does.

### Mini project A · The slot with its insulation

**The problem.** Ex_07.1 averaged the winding into one material. Here every one
of the 32 wires is resolved: a copper core, a thin enamel coat around it, and
air in the gaps between the wires. Copper conducts about 400 W/m·K, enamel
about 0.2 and air about 0.026, so the conductivity jumps by three to four
orders of magnitude at every wire edge. The heat is made in the copper only.
Each copper core is then almost the same temperature throughout, and the
temperature falls across the enamel and the air.

**Why a plain PINN is not enough.** The temperature is continuous at every
edge but its slope is not - the heat flow $k\,\partial\theta/\partial n$ is what
stays the same on both sides. A smooth network rounds that corner off, the
residual $\nabla\cdot(k\nabla\theta)$ does not exist on the edge, and the
enamel is so thin that random collocation points hardly ever land in it.

**What you build.** A domain-decomposed PINN: one network per material, each
smooth inside its own region, joined by interface terms in the loss that ask
for equal temperature and equal heat flow across every copper-enamel and
enamel-air edge. You choose how to sample the thin layers, how to weight the
interface terms, and whether the walls are soft or hard.

**The ground truth you get.** The resolved slot solved by finite differences,
on a grid with several nodes across the enamel and a conductivity set cell by
cell, for the currents of Ex_07.1, with its grid-refinement table.

**What you hand in.** Error maps against the ground truth, in each material;
the hot spot and its error; how well the heat flow matches across the edges;
and one comparison that matters to a designer - how far the averaged slot of
Ex_07.1 was from the resolved one, and whether its $k_{\text{eff}}$ was right.

### Mini project B · How old is the insulation?

**The problem.** Insulation ages. Heat degrades the enamel and the resin,
layers come apart and voids open, and the conductivity falls where that
happens - so the copper runs hotter, which ages the insulation faster. A
common rule of thumb says insulation life halves for every ten kelvin or so it
runs hotter. Nobody can open a machine in service to look. A few temperature
sensors in the slot, read at a few currents, are all there is.

**What you build.** An inverse PINN on the averaged slot of Ex_07.1, now with a
conductivity that varies over the slot: one network for the temperature
$\theta(x, y, I)$ and one for the conductivity $k(x, y)$, kept positive, fitted
together to the heat equation $\nabla\cdot(k\nabla\theta) + q = 0$ and to the
sensor readings. From the recovered $k$ you predict the hot spot and how far it
is from the insulation's class limit.

**The ground truth you get.** Slots with a degraded region hidden in them,
solved by finite differences: the sensor readings, with their positions,
currents and noise level, for you to fit; and the true conductivity and
temperature fields, for you to score yourself against afterwards. The cases
get harder - fewer sensors, more noise, a smaller degraded region.

**What you hand in.** The recovered conductivity against the true one, the
predicted hot spot against the true one, and how both depend on the number of
sensors and the noise. Say what the sensors cannot tell you: where a degraded
region could hide without any sensor noticing.

Next: **[Ex_07.2](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_00_environment_check.ipynb)**, where time enters and the question becomes how many initial
conditions a problem actually needs.